In [1]:
import Scope
from Scope.Read_Write import load_binary
from Scope.Classes_System import system
from Scope.Spin_Crossover.SCO_Classes import sco_system

In [2]:
sys = sco_system("ABITEM")


In [3]:
#sys.set_paths(create_folders=False, debug=1)
sys.sources_path = "/Users/sergivela/Documents/SCOPE/Database_SCO/4-Merged/ABITEM/"
sys.calcs_path = "/Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Calcs/ABITEM/"
sys.sys_path   = "/Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Systems/ABITEM/"
sys.sys_file   = "/Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Systems/ABITEM/ABITEM.npy"


In [4]:
sys

-------------------------------------
-- >>> SCOPE SCO-System Object >>> --
-------------------------------------
 Version               = 1.0
 Type                  = system
 Subtype               = sco_system
 Name                  = ABITEM
 Source Path           = /Users/sergivela/Documents/SCOPE/Database_SCO/4-Merged/ABITEM/
 Calculations Path     = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Calcs/ABITEM/
 System File Path      = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Systems/ABITEM/
 System File Name      = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Systems/ABITEM/ABITEM.npy


In [5]:
sys.load_multiple_cell2mol_folders("/Users/sergivela/Documents/SCOPE/Database_SCO/4-Merged/ABITEM/")

-------------------------------------
-- >>> SCOPE SCO-System Object >>> --
-------------------------------------
 Version               = 1.0
 Type                  = system
 Subtype               = sco_system
 Name                  = ABITEM
 Source Path           = /Users/sergivela/Documents/SCOPE/Database_SCO/4-Merged/ABITEM/
 Calculations Path     = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Calcs/ABITEM/
 System File Path      = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Systems/ABITEM/
 System File Name      = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Systems/ABITEM/ABITEM.npy

 # of Sources          = 2
     idx, type, name, formula               
     0: cell ABITEM01 H104-C92-N24-O4-S8-Fe4 
     1: cell ABITEM H104-C92-N24-O4-S8-Fe4 


In [6]:
sou = sys.sources[0]
sou

-----------------------------------
-- >>> SCOPE SCO-CELL Object >>> --
-----------------------------------
 Version               = 1.0
 Type                  = cell
 Name                  = ABITEM01
 Num Atoms             = 236
 Cell Parameters a:c   = [12.134, 11.987, 16.925]
 Cell Parameters al:ga = [90.0, 90.137, 90.0]
 # Molecules:          = 8
 With Formulae:                               
    0: H18-C20-N6-S2-Fe 
    1: H18-C20-N6-S2-Fe 
    2: H8-C3-O 
    3: H8-C3-O 
    4: H8-C3-O 
    5: H8-C3-O 
    6: H18-C20-N6-S2-Fe 
    7: H18-C20-N6-S2-Fe 
-------------------------------
 # of Ref Molecules:   = 2
 With Formulae:                                  
    0: H18-C20-N6-S2-Fe 
    1: H8-C3-O 
 Phase                 = LS
 HS Molar Fraction     = 0.0


In [7]:
istate = sou.add_state("initial")
istate.set_geometry(sou.labels, sou.coord)
istate.set_cell(sou.cell_vector, sou.cell_param)
istate.set_spin_config(spins=[2, 2, 0, 0], typ='metals', debug=1)

STATE.SET_SPIN_CONFIG: Preparing Spin Configuration for State initial
STATE.SET_SPIN_CONFIG: Received spins [2, 2, 0, 0]
STATE.SET_SPIN_CONFIG: Received typ metals
STATE.SET_SPIN_CONFIG: Setting spin=2 to the metal of molecule H18-C20-N6-S2-Fe in index 4
STATE.SET_SPIN_CONFIG: Formal SPIN is added to the first metal of the molecule: Fe
STATE.SET_SPIN_CONFIG: Setting spin=2 to the metal of molecule H18-C20-N6-S2-Fe in index 5
STATE.SET_SPIN_CONFIG: Formal SPIN is added to the first metal of the molecule: Fe
STATE.SET_SPIN_CONFIG: Setting spin=0 to the metal of molecule H18-C20-N6-S2-Fe in index 6
STATE.SET_SPIN_CONFIG: Formal SPIN is added to the first metal of the molecule: Fe
STATE.SET_SPIN_CONFIG: Setting spin=0 to the metal of molecule H18-C20-N6-S2-Fe in index 7
STATE.SET_SPIN_CONFIG: Formal SPIN is added to the first metal of the molecule: Fe


In [8]:
import os
from Scope.Classes_State       import find_state
from Scope.Classes_Environment import set_user
from Scope.Other               import get_metal_idxs, get_metal_species
from Scope.Elementdata         import ElementData 
from Scope.Software.Quantum_Espresso.QE_Functions import get_QE_data, get_pp
elemdatabase = ElementData()

In [9]:
metal_indices            = get_metal_idxs(istate.labels)     ## Indices of metal atoms in the structure 
metal_species            = get_metal_species(istate.labels)  ## 
elems                    = list(set(istate.labels))
pairs                    = get_QE_data(istate)

In [10]:
print(pairs)

[('C', 0), ('H', 0), ('O', 0), ('Fe', 2), ('S', 0), ('N', 0), ('Fe', 0)]


In [11]:
print(elems)

['H', 'Fe', 'C', 'N', 'O', 'S']


In [12]:
import Scope.Software.Quantum_Espresso 

In [13]:
test_input = """
&environment
   max_jobs         = 400
   max_procs        = 3200
   requested_procs  = 32
/

&options
   want_submit       = True
   ignore_submitted  = False
   overwrite_inputs  = False
   overwrite_logs    = False
/

&job_data
   branch       = 'Solid'
   target       = 'ref_crys'
   setup        = 'regular'
   suffix       = 'relax'
   keyword      = 'relax'

   istate       = 'initial'
   fstate       = 'relax'

   hierarchy    = 1
   requisites   = []
   constrains   = ['self']
   must_be_good = True
/

&qc_data
   software     = 'qe'
   version      = '7.0'
   jobtype      = 'relax'
   functional   = 'pbe'
   is_hubbard   = True
   is_grimme    = True
   uterm        = 2.35
   cutoff       = 25
   mix_beta     = 0.7
   pseudo       = '/home/svela/Programes/PP_Library'
   ## If "spin" is not specified, a default will be taken
   ## Maybe also PP_Library
/
"""

In [14]:
from Scope.Classes_Input import *
local_env   = input_data(content=test_input, section="&environment", isfile=False)
options     = input_data(content=test_input, section="&options", isfile=False)
job_data    = input_data(content=test_input, section="&job_data", isfile=False)
qc_data     = input_data(content=test_input, section="&qc_data", isfile=False)

In [15]:
sys = load_binary("/Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Systems/ABITEM/ABITEM.npy")

In [16]:
sys

-------------------------------------
-- >>> SCOPE SCO-System Object >>> --
-------------------------------------
 Version               = 1.0
 Type                  = system
 Subtype               = sco_system
 Name                  = ABITEM
 Source Path           = /Users/sergivela/Documents/SCOPE/Database_SCO/4-Merged/ABITEM/
 Calculations Path     = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Calcs/ABITEM/
 System File Path      = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Systems/ABITEM/
 System File Name      = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Systems/ABITEM/ABITEM.npy

 # of Sources          = 6
     idx, type, name, formula               
     0: cell ABITEM01 H104-C92-N24-O4-S8-Fe4 
     1: cell ABITEM H104-C92-N24-O4-S8-Fe4 
     2: cell ref_hs_cell H104-C92-N24-O4-S8-Fe4 
     3: cell ref_ls_cell H104-C92-N24-O4-S8-Fe4 
     4: specie ref_hs_mol H18-C20-N6-S2-Fe 
     5: specie ref_ls_mol H18-C20-N6-S2-Fe 


In [17]:
exists, this_branch = sys.find_branch("solid")
print(this_branch)

---------------------------------------------------
   >>> BRANCH                                      
---------------------------------------------------
 System                = ABITEM
---------------------------------------------------
 self.status           = active
 self.creation_time    = 20/10/2025 13:51:42
 self.creation_user    = sergivela
 self.path             = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Calcs/ABITEM/Solid/
 self.name             = Solid
 Num Recipes           = 2




In [18]:
this_branch.recipes

[---------------------------------------------------
    >>> >>> RECIPE                                  
 ---------------------------------------------------
  Source Name                 = ref_hs_cell
  Source Spin                 = None
  Source Phase                = HS
  Source Type                 = cell
  Source sub-Type             = sco_cell
 ---------------------------------------------------
  Recipe Name (from Source)   = ref_hs_cell
  Num Jobs                    = 1
 	Last Job Keyword     = relax
 	Last Job Hierarchy   = 1
 ,
 ---------------------------------------------------
    >>> >>> RECIPE                                  
 ---------------------------------------------------
  Source Name                 = ref_ls_cell
  Source Spin                 = None
  Source Phase                = LS
  Source Type                 = cell
  Source sub-Type             = sco_cell
 ---------------------------------------------------
  Recipe Name (from Source)   = ref_ls_cell
  Num

In [19]:
exists, this_recipe = this_branch.find_recipe("ref_hs_cell")
print(this_recipe)

---------------------------------------------------
   >>> >>> RECIPE                                  
---------------------------------------------------
 Source Name                 = ref_hs_cell
 Source Spin                 = None
 Source Phase                = HS
 Source Type                 = cell
 Source sub-Type             = sco_cell
---------------------------------------------------
 Recipe Name (from Source)   = ref_hs_cell
 Num Jobs                    = 1
	Last Job Keyword     = relax
	Last Job Hierarchy   = 1




In [20]:
exists, this_job = this_recipe.find_job("relax")
print(this_job)

---------------------------------------------------
   >>> >>> >>> JOB                                 
---------------------------------------------------
 Source Type           = cell
 Source Spin           = None
 Branch Name           = Solid
 Recipe Name           = ref_hs_cell
---------------------------------------------------
 Job path              = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Calcs/ABITEM/Solid/
 Job keyword           = relax
 Job hierarchy         = 1
 Job requisites        = []
 Job constrains        = ['self']
 Job setup             = regular
 Num Computations      = 1
----------------------------------------------------
 self.isregistered (Temp) = True
 self.isgood       (Temp) = False
 self.isfinished   (Temp) = False




In [21]:
exists, this_comp = this_job.find_computation(run_number=1)
print(this_comp)

---------------------------------------------------
   >>> >>> >>> >>> COMPUTATION                     
---------------------------------------------------
 Source Type           = cell
 Source sub-Type       = sco_cell
 Branch Name           = Solid
 Recipe Name           = ref_hs_cell
 Job Keyword           = relax
---------------------------------------------------
 Initial State         = initial
 Final State           = relax
 Comp software         = qe
 Comp index            = 1
 Comp step             = 1
 Comp run_number       = 1
 Comp keyword          = 
 Comp inp_path         = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Calcs/ABITEM/Solid/ABITEM_ref_hs_cell_relax_r1.input
 Comp out_path         = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Calcs/ABITEM/Solid/ABITEM_ref_hs_cell_relax_r1.out
 Comp isregistered     = False




In [22]:
this_comp.source.charge

In [23]:
exists, istate = this_comp.source.find_state(this_comp.qc_data.istate)
print(istate)

---------------------------------------------------
   STATE                                           
---------------------------------------------------
 Name                  = initial
 Source Name           = ref_hs_cell
 Source Type           = cell
 Labels                = Fe...
 Coord                 = [3.7430176, 3.9944543, 13.5094621]...
 Number of Complexes   = 4
 # Molecules:          = 8
 With Formulae:                               
    0: H8-C3-O 
    1: H8-C3-O 
    2: H8-C3-O 
    3: H8-C3-O 
    4: H18-C20-N6-S2-Fe 
    5: H18-C20-N6-S2-Fe 
    6: H18-C20-N6-S2-Fe 
    7: H18-C20-N6-S2-Fe 



In [24]:
istate.set_spin_config(spins=[2, 2, 0, 0], typ='metals', debug=1)

STATE.SET_SPIN_CONFIG: Preparing Spin Configuration for State initial
STATE.SET_SPIN_CONFIG: Received spins [2, 2, 0, 0]
STATE.SET_SPIN_CONFIG: Received typ metals
STATE.SET_SPIN_CONFIG: Setting spin=2 to the metal of molecule H18-C20-N6-S2-Fe in index 4
STATE.SET_SPIN_CONFIG: Formal SPIN is added to the first metal of the molecule: Fe
STATE.SET_SPIN_CONFIG: Setting spin=2 to the metal of molecule H18-C20-N6-S2-Fe in index 5
STATE.SET_SPIN_CONFIG: Formal SPIN is added to the first metal of the molecule: Fe
STATE.SET_SPIN_CONFIG: Setting spin=0 to the metal of molecule H18-C20-N6-S2-Fe in index 6
STATE.SET_SPIN_CONFIG: Formal SPIN is added to the first metal of the molecule: Fe
STATE.SET_SPIN_CONFIG: Setting spin=0 to the metal of molecule H18-C20-N6-S2-Fe in index 7
STATE.SET_SPIN_CONFIG: Formal SPIN is added to the first metal of the molecule: Fe


In [25]:
istate.get_atoms()
for at in istate.atoms:
    if at.spin != 0: print(at.label, at.spin, at.get_decorated_label("spin"))

Fe 2 Fe2
Fe 2 Fe2


In [26]:
species = get_QE_data(istate)

In [27]:
for idx, spec in enumerate(species):
    if spec[1] != 0: label = f"{spec[0]}{spec[1]}"
    else:            label = spec[0]
    weight = elemdatabase.elementweight[spec[0]]
    print(f"{label} {weight:6.4f} ")

C 12.0106 
H 1.0075 
O 15.9990 
Fe2 55.8450 
S 32.0675 
N 14.0065 
Fe 55.8450 


In [28]:
this_comp.inp_path

'/Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Calcs/ABITEM/Solid/ABITEM_ref_hs_cell_relax_r1.input'

In [29]:
from Scope.Software.Quantum_Espresso import *
gen_QE_input(this_comp, debug=1)

GEN_QE_INPUT: system_type: cell
GEN_QE_INPUT: creating input in path: /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Calcs/ABITEM/Solid/ABITEM_ref_hs_cell_relax_r1.input
GEN_QE_INPUT: Received cell with this data:
    species:  [('C', 0), ('H', 0), ('O', 0), ('Fe', 2), ('S', 0), ('N', 0), ('Fe', 0)]
    metal_species: ['Fe']
    metal_indices: [0, 1, 6, 7]
    ismagnetic:    True


In [31]:
for idx, at in enumerate(istate.atoms):
    print(at.label)
    if idx in metal_indices:
        if at.spin == 0: l = at.label
        else:            l = at.get_decorated_label(typ="spin") 
        print(f"{l:4}")

C
C   
H
H   
H
H
C
H
H
H   
C
C   
O
H
H
H
C
H
H
H
C
H
H
C
O
H
H
H
C
H
H
H
C
H
C
O
H
H
H
H
C
H
H
H
C
H
C
O
H
H
H
H
Fe
S
S
N
N
N
N
N
N
C
H
C
H
C
C
C
C
H
C
H
C
C
C
H
C
H
C
C
C
H
C
H
C
H
C
C
H
H
C
H
C
H
H
H
H
H
H
Fe
S
S
N
N
N
N
N
N
C
H
C
H
C
H
C
H
C
C
H
H
C
H
C
H
C
C
H
H
C
H
C
H
C
H
C
H
C
C
H
H
C
C
C
H
C
H
Fe
S
S
N
N
N
N
N
N
C
H
C
H
C
H
C
H
C
C
H
H
C
H
C
H
C
C
H
H
C
H
C
H
C
H
C
H
C
C
H
H
C
C
C
H
C
H
Fe
S
S
N
N
N
N
N
N
C
H
C
H
C
C
C
C
H
C
H
C
C
C
H
C
H
C
C
C
H
C
H
C
H
C
H
C
H
C
C
H
H
H
H
H
H
H
